In [1]:
import csv
import math
import os
import shutil
import time
from dataclasses import dataclass
from typing import Optional, Sequence

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from torchmetrics.image.fid import FrechetInceptionDistance

from configs import UNetConfig
from models.unet import UNet


In [2]:
class MultiResDriftConfig:
    pass

@dataclass
class EvaluationConfig:
    # Checkpoint
    model_version: str = "nystrom_conditional_cfg_M512"
    checkpoint_name: str = "drift_step0050000.pt"

    # Alpha values for the conditional-CFG sweep.
    #
    # alpha=1.0 is the class-conditional model without extra guidance.
    # Small values around 1.0 should be evaluated densely.
    alphas: Sequence[float] = (
        1.00,
        # 1.05,
        # 1.10,
        # 1.20,
        # 1.50,
        # 2.00,
    )

    # Historical protocol used by your existing evaluator:
    #   10,000 generated images against CIFAR-10's 10,000 test images.
    #
    # Keep this True when comparing against your old recorded scores.
    legacy_test_10k: bool = True

    # For a final thesis table, consider:
    #
    #   legacy_test_10k = False
    #   n_fid = 50_000
    #
    # This uses the 50,000-image CIFAR-10 training set as the reference.
    n_fid: int = 10_000

    fid_batch_size: int = 256
    generation_batch_size: int = 512
    data_loader_workers: int = 4

    # Identical random seed for every alpha gives paired comparisons:
    # each alpha sees the same z and class labels.
    generation_seed: int = 12_345

    # Delete previously generated PNGs before evaluating an alpha.
    # This prevents stale files from entering the FID calculation.
    regenerate_images: bool = True

    # Preserve the preprocessing behavior of the old evaluator.
    #
    # TorchMetrics already resizes images internally, so False is also valid,
    # but changing it would make the new numbers less directly comparable
    # with your old numbers.
    legacy_explicit_resize_299: bool = True

    # Sampling benchmark
    benchmark_samples: int = 1_024
    benchmark_batch_size: int = 256
    benchmark_warmup_batches: int = 5

    # Grid generation
    grid_samples_per_class: int = 8
    unconditional_grid_size: int = 64

cfg = EvaluationConfig()

# ============================================================================
# Device
# ============================================================================

device = torch.device("cuda:6")
print(f"Evaluation device: {device}")


Evaluation device: cuda:6


In [3]:
# ============================================================================
# General helpers
# ============================================================================


def config_value(config, name, default):
    """Read a field from either a dict or a dataclass-like object."""
    if config is None:
        return default
    if isinstance(config, dict):
        return config.get(name, default)
    return getattr(config, name, default)


def strip_compile_prefix(state_dict):
    """Remove torch.compile's _orig_mod prefix without changing other keys."""
    cleaned = {}

    for key, value in state_dict.items():
        while key.startswith("_orig_mod."):
            key = key[len("_orig_mod."):]
        cleaned[key] = value

    return cleaned


def alpha_tag(alpha: Optional[float]) -> str:
    """Filesystem-safe representation of an alpha value."""
    if alpha is None:
        return "none"
    return f"{float(alpha):.3f}".replace(".", "p")


def clear_png_directory(path):
    """Create a directory and remove existing PNG files from it."""
    os.makedirs(path, exist_ok=True)

    for filename in os.listdir(path):
        if filename.lower().endswith(".png"):
            os.remove(os.path.join(path, filename))


def count_png_files(path):
    if not os.path.isdir(path):
        return 0

    return sum(
        filename.lower().endswith(".png")
        for filename in os.listdir(path)
    )


In [4]:
# ============================================================================
# Model loading
# ============================================================================


def load_checkpoint_file(ckpt_path):
    """Load a trusted local training checkpoint on CPU.

    Loading on CPU avoids initially materializing all checkpoint tensors on
    the GPU. weights_only=False is needed because the standalone checkpoint
    contains config dataclass instances.
    """
    try:
        return torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # Compatibility with older PyTorch versions without weights_only.
        return torch.load(
            ckpt_path,
            map_location="cpu",
        )


def load_model(ckpt_path, device):
    """Load old and new drifting checkpoints.

    Supported cases:
      - unconditional checkpoints;
      - ordinary class-conditional checkpoints;
      - older OT-CFG checkpoints;
      - new conditional Nyström-CFG checkpoints;
      - torch.compile-prefixed state dictionaries.
    """
    print(f"Loading checkpoint: {ckpt_path}")

    checkpoint = load_checkpoint_file(ckpt_path)

    checkpoint_config = checkpoint.get("config", {})
    checkpoint_unet_cfg = config_value(
        checkpoint_config,
        "unet",
        {},
    )
    checkpoint_drift_cfg = config_value(
        checkpoint_config,
        "drift",
        {},
    )

    default_unet_cfg = UNetConfig()

    # Prefer EMA weights when available.
    state_dict = checkpoint.get(
        "ema",
        checkpoint["model"],
    )
    state_dict = strip_compile_prefix(state_dict)

    has_class_embedding = "class_embed.weight" in state_dict
    has_cfg_embedding = any(
        key.startswith("cfg_embed.")
        for key in state_dict
    )

    configured_num_classes = int(
        config_value(
            checkpoint_drift_cfg,
            "num_classes",
            config_value(
                checkpoint_unet_cfg,
                "num_classes",
                0,
            ),
        )
    )

    if has_class_embedding:
        inferred_num_classes = int(
            state_dict["class_embed.weight"].shape[0]
        )
        num_classes = (
            configured_num_classes
            if configured_num_classes > 0
            else inferred_num_classes
        )
    else:
        num_classes = 0

    # Use literal defaults where older UNetConfig definitions might not
    # contain fields such as image_size.
    image_size = int(
        config_value(
            checkpoint_unet_cfg,
            "image_size",
            getattr(default_unet_cfg, "image_size", 32),
        )
    )

    model = UNet(
        in_ch=config_value(
            checkpoint_unet_cfg,
            "in_ch",
            getattr(default_unet_cfg, "in_ch", 3),
        ),
        out_ch=config_value(
            checkpoint_unet_cfg,
            "out_ch",
            getattr(default_unet_cfg, "out_ch", 3),
        ),
        base_ch=config_value(
            checkpoint_unet_cfg,
            "base_ch",
            getattr(default_unet_cfg, "base_ch", 128),
        ),
        ch_mult=tuple(
            config_value(
                checkpoint_unet_cfg,
                "ch_mult",
                getattr(default_unet_cfg, "ch_mult", (1, 2, 2, 2)),
            )
        ),
        num_res_blocks=int(
            config_value(
                checkpoint_unet_cfg,
                "num_res_blocks",
                getattr(default_unet_cfg, "num_res_blocks", 2),
            )
        ),
        attn_resolutions=tuple(
            config_value(
                checkpoint_unet_cfg,
                "attn_resolutions",
                getattr(default_unet_cfg, "attn_resolutions", (16,)),
            )
        ),
        # Dropout has no parameters, so setting it to zero is safe at eval.
        dropout=0.0,
        num_heads=int(
            config_value(
                checkpoint_unet_cfg,
                "num_heads",
                getattr(default_unet_cfg, "num_heads", 4),
            )
        ),
        num_classes=num_classes,
        image_size=image_size,
    )

    incompatible = model.load_state_dict(
        state_dict,
        strict=False,
    )

    # Missing CFG keys are allowed for older conditional architectures that
    # did not contain cfg_embed. Other missing model parameters are suspicious.
    unexpected_missing = [
        key for key in incompatible.missing_keys
        if not key.startswith("cfg_embed.")
    ]

    if unexpected_missing:
        raise RuntimeError(
            "Checkpoint is missing required model parameters:\n"
            + "\n".join(unexpected_missing)
        )

    if incompatible.unexpected_keys:
        raise RuntimeError(
            "Checkpoint contains unexpected model parameters:\n"
            + "\n".join(incompatible.unexpected_keys)
        )

    model = model.to(device)
    model.eval()

    # ---------------------------------------------------------------------
    # Determine CFG capabilities
    # ---------------------------------------------------------------------

    # New conditional Nyström training configuration.
    new_cfg_enabled = bool(
        config_value(
            checkpoint_drift_cfg,
            "cfg_enabled",
            False,
        )
    )

    new_default_alpha = float(
        config_value(
            checkpoint_drift_cfg,
            "eval_cfg_scale",
            1.0,
        )
    )

    # Older OT-CFG configuration.
    loss_mode = str(
        config_value(
            checkpoint_drift_cfg,
            "loss_mode",
            "standard",
        )
    ).lower()

    old_ot_cfg_enabled = bool(
        config_value(
            checkpoint_drift_cfg,
            "ot_use_new_cfg",
            False,
        )
    )

    old_ot_default_alpha = float(
        config_value(
            checkpoint_drift_cfg,
            "ot_cfg_max",
            1.0,
        )
    )

    supports_cfg = (
        num_classes > 0
        and has_cfg_embedding
        and (
            new_cfg_enabled
            or (loss_mode == "ot" and old_ot_cfg_enabled)
        )
    )

    if new_cfg_enabled:
        default_alpha = new_default_alpha
    elif old_ot_cfg_enabled:
        default_alpha = old_ot_default_alpha
    else:
        default_alpha = 1.0

    # Evaluation metadata. These are ordinary Python attributes and are not
    # part of the state dictionary.
    model._eval_num_classes = num_classes
    model._eval_supports_cfg = supports_cfg
    model._eval_has_cfg_embedding = has_cfg_embedding
    model._eval_cfg_scale = default_alpha
    model._eval_checkpoint_step = int(
        checkpoint.get("step", -1)
    )

    print(f"  Checkpoint step: {model._eval_checkpoint_step}")
    print(f"  Number of classes: {num_classes}")
    print(f"  CFG embedding present: {has_cfg_embedding}")
    print(f"  Checkpoint trained with CFG: {supports_cfg}")
    print(f"  Default evaluation alpha: {default_alpha:.3f}")

    return model


In [5]:
# ============================================================================
# Sampling
# ============================================================================


def make_cuda_generator(device, seed):
    """Create a deterministic generator on the sampling device."""
    if device.type == "cuda":
        generator = torch.Generator(device=device)
    else:
        generator = torch.Generator()

    generator.manual_seed(int(seed))
    return generator


@torch.inference_mode()
def sample_drifting_batch(
    model,
    n,
    device,
    generator,
    start_idx=0,
    alpha=None,
    class_labels=None,
):
    """Sample one batch from old or new drifting models.

    For the new model, alpha is passed directly to the learned alpha
    conditioning. This remains one forward pass; it is not two-pass diffusion
    classifier-free guidance.
    """
    n = int(n)

    z = torch.randn(
        n,
        3,
        32,
        32,
        device=device,
        generator=generator,
    )

    num_classes = int(
        getattr(model, "_eval_num_classes", 0)
    )
    supports_cfg = bool(
        getattr(model, "_eval_supports_cfg", False)
    )

    if num_classes > 0:
        if class_labels is None:
            class_labels = (
                torch.arange(
                    start_idx,
                    start_idx + n,
                    device=device,
                    dtype=torch.long,
                )
                % num_classes
            )
        else:
            class_labels = class_labels.to(
                device=device,
                dtype=torch.long,
            )

        if alpha is None:
            alpha = float(
                getattr(model, "_eval_cfg_scale", 1.0)
            )

        if supports_cfg:
            cfg_scale = torch.full(
                (n,),
                float(alpha),
                device=device,
                dtype=torch.float32,
            )

            samples = model(
                z,
                class_labels=class_labels,
                cfg_scale=cfg_scale,
            )
        else:
            # Backwards-compatible class-conditional model without CFG.
            samples = model(
                z,
                class_labels=class_labels,
            )
    else:
        samples = model(z)

    return samples.clamp(-1, 1)


@torch.inference_mode()
def generate_sample_grid(
    model,
    output_path,
    device,
    alpha=None,
    seed=1234,
    samples_per_class=8,
    unconditional_count=64,
):
    """Generate a balanced visual grid."""
    generator = make_cuda_generator(
        device,
        seed,
    )

    num_classes = int(
        getattr(model, "_eval_num_classes", 0)
    )

    if num_classes > 0:
        # Rows correspond to classes when nrow=samples_per_class.
        labels = torch.arange(
            num_classes,
            device=device,
            dtype=torch.long,
        ).repeat_interleave(samples_per_class)

        samples = sample_drifting_batch(
            model=model,
            n=labels.numel(),
            device=device,
            generator=generator,
            class_labels=labels,
            alpha=alpha,
        )
        nrow = samples_per_class
    else:
        samples = sample_drifting_batch(
            model=model,
            n=unconditional_count,
            device=device,
            generator=generator,
            alpha=None,
        )
        nrow = int(math.sqrt(unconditional_count))

    grid = make_grid(
        samples,
        nrow=nrow,
        normalize=True,
        value_range=(-1, 1),
        padding=2,
    )

    save_image(grid, output_path)


@torch.inference_mode()
def generate_fid_images(
    model,
    n_images,
    output_dir,
    device,
    alpha=None,
    batch_size=512,
    seed=12345,
    clear_existing=True,
):
    """Generate deterministic PNGs for FID evaluation.

    Reusing the same seed for every alpha gives every alpha exactly the same
    latent sequence and class-label sequence.
    """
    if clear_existing:
        clear_png_directory(output_dir)
    else:
        os.makedirs(output_dir, exist_ok=True)

    generator = make_cuda_generator(
        device,
        seed,
    )

    generated_count = 0

    while generated_count < n_images:
        current_batch_size = min(
            batch_size,
            n_images - generated_count,
        )

        samples = sample_drifting_batch(
            model=model,
            n=current_batch_size,
            device=device,
            generator=generator,
            start_idx=generated_count,
            alpha=alpha,
        )

        # Convert once to [0, 1].
        samples = (
            (samples.float() + 1.0) * 0.5
        ).clamp(0, 1)

        for local_index, image in enumerate(samples):
            global_index = generated_count + local_index

            save_image(
                image,
                os.path.join(
                    output_dir,
                    f"{global_index:06d}.png",
                ),
            )

        generated_count += current_batch_size

        if (
            generated_count % 5_000 == 0
            or generated_count == n_images
        ):
            print(
                f"    Generated {generated_count}/{n_images} images"
            )

    actual_count = count_png_files(output_dir)

    if actual_count != n_images:
        raise RuntimeError(
            f"Expected {n_images} PNGs in {output_dir}, "
            f"but found {actual_count}."
        )


In [6]:
# ============================================================================
# Sampling benchmark
# ============================================================================

@torch.inference_mode()
def benchmark_sampling(
    model,
    n,
    device,
    alpha=None,
    batch_size=256,
    warmup_batches=5,
    seed=9876,
):
    """Measure sampling throughput without constructing one huge batch."""
    warmup_generator = make_cuda_generator(
        device,
        seed,
    )

    for warmup_index in range(warmup_batches):
        sample_drifting_batch(
            model=model,
            n=min(16, batch_size),
            device=device,
            generator=warmup_generator,
            start_idx=warmup_index * 16,
            alpha=alpha,
        )

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    benchmark_generator = make_cuda_generator(
        device,
        seed + 1,
    )

    start_time = time.perf_counter()
    generated_count = 0

    while generated_count < n:
        current_batch_size = min(
            batch_size,
            n - generated_count,
        )

        sample_drifting_batch(
            model=model,
            n=current_batch_size,
            device=device,
            generator=benchmark_generator,
            start_idx=generated_count,
            alpha=alpha,
        )

        generated_count += current_batch_size

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    elapsed = time.perf_counter() - start_time
    return n / elapsed


# ============================================================================
# Reference images
# ============================================================================


def save_cifar10_reference_images(
    root_dir,
    n_images,
    use_train_split,
):
    """Save an exact CIFAR-10 split as PNG files in [0, 1]."""
    clear_png_directory(root_dir)

    dataset = datasets.CIFAR10(
        root="./data",
        train=use_train_split,
        download=True,
        transform=transforms.ToTensor(),
    )

    if n_images > len(dataset):
        raise ValueError(
            f"Requested {n_images} real images, but the selected CIFAR-10 "
            f"split contains only {len(dataset)}."
        )

    for index in range(n_images):
        image, _ = dataset[index]

        save_image(
            image,
            os.path.join(
                root_dir,
                f"{index:06d}.png",
            ),
        )

        if (index + 1) % 10_000 == 0:
            print(
                f"    Saved {index + 1}/{n_images} reference images"
            )


In [7]:
# ============================================================================
# FID
# ============================================================================

class PNGDataset(Dataset):
    def __init__(
        self,
        root_dir,
        explicit_resize_299=False,
    ):
        self.image_files = sorted(
            os.path.join(root_dir, filename)
            for filename in os.listdir(root_dir)
            if filename.lower().endswith(".png")
        )

        operations = []

        if explicit_resize_299:
            # Included only to reproduce the existing evaluator exactly.
            operations.append(
                transforms.Resize(
                    (299, 299),
                    antialias=True,
                )
            )

        operations.append(transforms.ToTensor())
        self.transform = transforms.Compose(operations)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image = Image.open(
            self.image_files[index]
        ).convert("RGB")

        return self.transform(image)


def update_fid_from_directory(
    fid,
    directory,
    real,
    batch_size,
    num_workers,
    explicit_resize_299,
):
    dataset = PNGDataset(
        directory,
        explicit_resize_299=explicit_resize_299,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(num_workers > 0),
        drop_last=False,
    )

    for batch in loader:
        fid.update(
            batch.to(
                device,
                non_blocking=True,
            ),
            real=real,
        )


def create_fid_metric():
    """Construct FID in float64 for more stable covariance computation."""
    metric = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    # TorchMetrics recommends float64 for greater numerical stability.
    metric.set_dtype(torch.float64)
    return metric


In [8]:
# ============================================================================
# Evaluation
# ============================================================================


model_version = cfg.model_version

training_dir = os.path.join(
    "outputs",
    "training",
    model_version,
)

checkpoint_path = os.path.join(
    training_dir,
    "checkpoints",
    cfg.checkpoint_name,
)

out_dir = os.path.join(
    "outputs",
    "evaluation",
    model_version,
    os.path.splitext(cfg.checkpoint_name)[0],
)

os.makedirs(out_dir, exist_ok=True)

drift_model = load_model(
    checkpoint_path,
    device,
)

num_classes = int(
    getattr(drift_model, "_eval_num_classes", 0)
)
supports_cfg = bool(
    getattr(drift_model, "_eval_supports_cfg", False)
)

# Do not run a meaningless sweep for unconditional or non-CFG models.
if num_classes == 0:
    evaluation_alphas = [None]
elif supports_cfg:
    evaluation_alphas = list(cfg.alphas)
else:
    evaluation_alphas = [1.0]

print("\nEvaluation settings:")
print(f"  Conditional: {num_classes > 0}")
print(f"  CFG sweep enabled: {supports_cfg}")
print(f"  Alpha values: {evaluation_alphas}")
print(f"  FID image count: {cfg.n_fid}")
print(f"  Shared generation seed: {cfg.generation_seed}")

if cfg.legacy_test_10k:
    if cfg.n_fid > 10_000:
        raise ValueError(
            "The CIFAR-10 test split contains only 10,000 images. "
            "Use legacy_test_10k=False for a 50,000-image reference."
        )

    reference_uses_train_split = False
    reference_protocol_name = "cifar10_test"
else:
    reference_uses_train_split = True
    reference_protocol_name = "cifar10_train"


Loading checkpoint: outputs/training/nystrom_conditional_cfg_M512/checkpoints/drift_step0050000.pt


  Checkpoint step: 50000
  Number of classes: 10
  CFG embedding present: True
  Checkpoint trained with CFG: True
  Default evaluation alpha: 1.000

Evaluation settings:
  Conditional: True
  CFG sweep enabled: True
  Alpha values: [1.0]
  FID image count: 10000
  Shared generation seed: 12345


In [9]:
# --------------------------------------------------------------------------
# Sample grids
# --------------------------------------------------------------------------

print("\nGenerating sample grids...")

for alpha in evaluation_alphas:
    if alpha is None:
        filename = "drift_samples_unconditional.png"
    else:
        filename = (
            f"drift_samples_alpha_{alpha_tag(alpha)}.png"
        )

    generate_sample_grid(
        model=drift_model,
        output_path=os.path.join(out_dir, filename),
        device=device,
        alpha=alpha,
        seed=cfg.generation_seed,
        samples_per_class=cfg.grid_samples_per_class,
        unconditional_count=cfg.unconditional_grid_size,
    )

    print(f"  Saved {filename}")



Generating sample grids...


  Saved drift_samples_alpha_1p000.png


In [10]:
# --------------------------------------------------------------------------
# Sampling speed
# --------------------------------------------------------------------------

print("\nBenchmarking sampling speed...")

benchmark_rows = []

for alpha in evaluation_alphas:
    images_per_second = benchmark_sampling(
        model=drift_model,
        n=cfg.benchmark_samples,
        device=device,
        alpha=alpha,
        batch_size=cfg.benchmark_batch_size,
        warmup_batches=cfg.benchmark_warmup_batches,
    )

    alpha_display = (
        "unconditional"
        if alpha is None
        else f"{alpha:.3f}"
    )

    print(
        f"  Alpha {alpha_display:>13s}: "
        f"{images_per_second:.1f} images/sec"
    )

    benchmark_rows.append({
        "alpha": alpha_display,
        "images_per_second": images_per_second,
    })



Benchmarking sampling speed...


  Alpha         1.000: 2403.9 images/sec


In [11]:
# --------------------------------------------------------------------------
# Real/reference directory
# --------------------------------------------------------------------------

reference_dir = os.path.join(
    out_dir,
    f"fid_ref_{reference_protocol_name}_{cfg.n_fid}",
)

reference_count = count_png_files(reference_dir)

if reference_count != cfg.n_fid:
    print(
        f"\nSaving {cfg.n_fid} reference CIFAR-10 images..."
    )

    save_cifar10_reference_images(
        root_dir=reference_dir,
        n_images=cfg.n_fid,
        use_train_split=reference_uses_train_split,
    )
else:
    print(
        f"\nUsing existing reference directory with "
        f"{reference_count} images."
    )



Saving 10000 reference CIFAR-10 images...


    Saved 10000/10000 reference images


In [12]:
# --------------------------------------------------------------------------
# Initialize real FID statistics once
# --------------------------------------------------------------------------

print("\nExtracting real-image FID statistics...")

fid_metric = create_fid_metric()

update_fid_from_directory(
    fid=fid_metric,
    directory=reference_dir,
    real=True,
    batch_size=cfg.fid_batch_size,
    num_workers=cfg.data_loader_workers,
    explicit_resize_299=cfg.legacy_explicit_resize_299,
)



Extracting real-image FID statistics...


In [13]:
# --------------------------------------------------------------------------
# Generate and evaluate each alpha
# --------------------------------------------------------------------------

print("\nEvaluating generated distributions...")

results = []

for alpha in evaluation_alphas:
    if alpha is None:
        run_name = "unconditional"
    else:
        run_name = f"alpha_{alpha_tag(alpha)}"

    fake_dir = os.path.join(
        out_dir,
        f"fid_fake_{run_name}",
    )

    should_generate = (
        cfg.regenerate_images
        or count_png_files(fake_dir) != cfg.n_fid
    )

    print(f"\n  Run: {run_name}")

    if should_generate:
        print("    Generating samples...")

        generation_start = time.perf_counter()

        generate_fid_images(
            model=drift_model,
            n_images=cfg.n_fid,
            output_dir=fake_dir,
            device=device,
            alpha=alpha,
            batch_size=cfg.generation_batch_size,
            seed=cfg.generation_seed,
            clear_existing=True,
        )

        if device.type == "cuda":
            torch.cuda.synchronize(device)

        generation_time = (
            time.perf_counter() - generation_start
        )
    else:
        print(
            f"    Reusing {count_png_files(fake_dir)} existing images."
        )
        generation_time = float("nan")

    print("    Extracting generated-image FID statistics...")

    update_fid_from_directory(
        fid=fid_metric,
        directory=fake_dir,
        real=False,
        batch_size=cfg.fid_batch_size,
        num_workers=cfg.data_loader_workers,
        explicit_resize_299=cfg.legacy_explicit_resize_299,
    )

    fid_value = float(
        fid_metric.compute().detach().cpu().item()
    )

    print(f"    FID: {fid_value:.4f}")

    alpha_result = (
        "unconditional"
        if alpha is None
        else f"{float(alpha):.6f}"
    )

    results.append({
        "alpha": alpha_result,
        "fid": fid_value,
        "generation_time_s": generation_time,
        "n_images": cfg.n_fid,
        "reference_protocol": reference_protocol_name,
        "reference_count": cfg.n_fid,
        "seed": cfg.generation_seed,
    })

    # reset_real_features=False preserves the already-computed real
    # statistics while clearing fake statistics for the next alpha.
    fid_metric.reset()



Evaluating generated distributions...

  Run: alpha_1p000
    Generating samples...


    Generated 10000/10000 images
    Extracting generated-image FID statistics...
    FID: 20.7691


In [ ]:
# --------------------------------------------------------------------------
# Save results
# --------------------------------------------------------------------------

results_path = os.path.join(
    out_dir,
    "alpha_sweep_results.csv",
)

with open(results_path, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "alpha",
            "fid",
            "generation_time_s",
            "n_images",
            "reference_protocol",
            "reference_count",
            "seed",
        ],
    )
    writer.writeheader()
    writer.writerows(results)

benchmark_path = os.path.join(
    out_dir,
    "sampling_benchmark.csv",
)

with open(benchmark_path, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "alpha",
            "images_per_second",
        ],
    )
    writer.writeheader()
    writer.writerows(benchmark_rows)


print("\n" + "=" * 72)
print("ALPHA SWEEP RESULTS")
print("=" * 72)

for result in sorted(
    results,
    key=lambda row: row["fid"],
):
    print(
        f"alpha={result['alpha']:>13s} | "
        f"FID={result['fid']:.4f}"
    )

best_result = min(
    results,
    key=lambda row: row["fid"],
)

print("-" * 72)
print(
    f"Best alpha: {best_result['alpha']} | "
    f"FID: {best_result['fid']:.4f}"
)
print(f"Results saved to: {results_path}")

"""
ot_old_cfg; 60k steps 31.2 FID; nystrom: 50k stpes 25.2 FID, standard: 47.3; ot_newton: 50k steps 36.1 FID; ot_newton_nystrom: 50k steps 37.1 FID

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg 100k steps - 6.5 hours
========================================================================
alpha=     1.000000 | FID=14.4736
alpha=     1.050000 | FID=14.5221
alpha=     1.100000 | FID=14.5555
alpha=     1.200000 | FID=14.7157
alpha=     1.500000 | FID=15.7295
alpha=     2.000000 | FID=18.2417

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg 50k steps - 3.3 hours
========================================================================
alpha=     1.000000 | FID=20.1520

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_sample32 50k steps - 3.2 hours
========================================================================
alpha=     1.000000 | FID=19.2211

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_sample64 50k steps - 3.2 hours
========================================================================
alpha=     1.000000 | FID=14.5890

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_sample16 50k steps - 3.3 hours
========================================================================
alpha=     1.000000 | FID=17.0195

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_ratio 50k steps - 3.4 hours
========================================================================
alpha=     1.000000 | FID=15.4427

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_m82 50k steps - 3.2 hours
========================================================================
alpha=     1.000000 | FID=15.7679

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_m164 50k steps - 3.1 hours
========================================================================
alpha=     1.000000 | FID=17.3030

========================================================================
ALPHA SWEEP RESULTS - nystrom_conditional_cfg_m512 50k steps - 6.4 hours
========================================================================
alpha=     1.000000 | FID=20.7691

"""





ALPHA SWEEP RESULTS
alpha=     1.000000 | FID=20.7691
------------------------------------------------------------------------
Best alpha: 1.000000 | FID: 20.7691
Results saved to: outputs/evaluation/nystrom_conditional_cfg_M512/drift_step0050000/alpha_sweep_results.csv


'\not_old_cfg; 60k steps 31.2 FID; nystrom: 50k stpes 25.2 FID, standard: 47.3; ot_newton: 50k steps 36.1 FID; ot_newton_nystrom: 50k steps 37.1 FID\n\n========================================================================\nALPHA SWEEP RESULTS - nystrom_conditional_cfg 100k steps\n========================================================================\nalpha=     1.000000 | FID=14.4736\nalpha=     1.050000 | FID=14.5221\nalpha=     1.100000 | FID=14.5555\nalpha=     1.200000 | FID=14.7157\nalpha=     1.500000 | FID=15.7295\nalpha=     2.000000 | FID=18.2417\n\n========================================================================\nALPHA SWEEP RESULTS - nystrom_conditional_cfg 50k steps\n========================================================================\nalpha=     1.000000 | FID=20.1520\n\n========================================================================\nALPHA SWEEP RESULTS - nystrom_conditional_cfg_cv_sample32 50k steps\n=============================================